# Luminancia

## Media de luminancia

Promedia los valores de luminancia de todos los píxeles para obtener el nivel de brillo general de la imagen.

In [1]:
import numpy as np
from PIL import Image

def obtener_media_luminancia(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    M, N = arr.shape
    return (1 / (M * N)) * np.sum(arr)

In [2]:
print(obtener_media_luminancia('cute.png'))

141.69573459253738


## Varianza de luminancia

Mide qué tan uniformemente distribuida está la luminancia. Valores altos = alto contraste; valores bajos = uniformidad.

In [3]:
import numpy as np
from PIL import Image

def obtener_varianza_luminancia(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    M, N = arr.shape
    mu = np.mean(arr)
    return (1 / (M * N)) * np.sum((arr - mu) ** 2)

In [4]:
print(obtener_varianza_luminancia('cute.png'))

6379.67791995716


## Histograma de intensidades

Frecuencia de cada nivel de intensidad (0–255) en la imagen. Base para diagnóstico de exposición y contraste.

In [5]:
import numpy as np
from PIL import Image

def obtener_histograma_intensidades(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img)
    hist = np.zeros(256, dtype=int)
    for k in range(256):
        hist[k] = np.sum(arr == k)
    hist_norm = hist / arr.size  # p(rk) = nk / MN
    return hist, hist_norm

In [9]:
hist, hist_norm = obtener_histograma_intensidades('cute.png')
print(hist)

[19414  7952  1316   681   532   391   285   265   183   162   179   167
   156   176   207   257   253   271   282   393   827  1566  6973  1362
   194   113   106    71    78    55    69    71    56    47    45    49
    51    58    54    54    50    48    59    42    63    41    43    41
    36    28    31    43    46    39    39    37    29    34    22    39
    37    31    35    35    27    36    29    40    39    35    29    22
    39    35    36    38    37    29    43    45    35    38    26    43
    48    45    40    46    44    36    38    53    55    37    39    42
    36    44    44    44    39    42    54    43    42    59    54    56
    76   117   196   256   483   344   138   112   122   141   106   129
   146   125   122   134   112   123   119   101   120   102   127    90
   120   117   135   123   145   103   104    99   132   126   109   116
   104   115   120   129    98   116   126   174   207   318   445   606
  3350  9924 31609 32591  1349   136   175   165   

## Rango dinámico

Diferencia entre el valor máximo y mínimo de luminancia. Indica qué tan bien se captura el espectro completo de brillos.

In [20]:
import numpy as np
from PIL import Image

def obtener_rango_dinamico(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    return float(arr.max() - arr.min())

In [23]:
print(obtener_rango_dinamico('Imagenes/cute.jpg'))

255.0


## Desviación estándar de brillo

Raíz cuadrada de la varianza de luminancia. Representa el contraste global típico de la imagen.

In [10]:
import numpy as np
from PIL import Image

def obtener_desv_estandar_brillo(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    M, N = arr.shape
    mu = np.mean(arr)
    return np.sqrt((1 / (M * N)) * np.sum((arr - mu) ** 2))

In [11]:
print(obtener_desv_estandar_brillo('cute.png'))

79.8728860124458


## Contraste local basado en luminancia

Compara la luminancia de cada píxel con la media de su vecindario local, normalizada por dicha media.

In [15]:
import numpy as np
from PIL import Image
from scipy.ndimage import uniform_filter

def obtener_contraste_local_luminancia(img_path, ventana=5):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    mu_N = uniform_filter(arr, size=ventana)
    with np.errstate(divide='ignore', invalid='ignore'):
        CL = np.where(mu_N != 0, (arr - mu_N) / mu_N, 0)
    return CL

In [16]:
print(obtener_contraste_local_luminancia('cute.png'))

[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  1.82190445e-16
   1.82190445e-16  1.82190445e-16]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  3.12326477e-16
   3.12326477e-16  3.12326477e-16]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -4.68489716e-16
  -4.68489716e-16 -4.68489716e-16]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -4.68489716e-16
  -4.68489716e-16 -4.68489716e-16]]


# Contraste 

## Contraste RMS

Desviación estándar de los valores de píxel. Mide la variación de luminancia como métrica de contraste global.

In [24]:
import numpy as np
from PIL import Image

def obtener_contraste_rms(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    M, N = arr.shape
    mu = np.mean(arr)
    return np.sqrt((1 / (M * N)) * np.sum((arr - mu) ** 2))

In [25]:
print(obtener_contraste_rms('cute.png'))

79.8728860124458


## Contraste de Michelson

Contraste modular entre máximo y mínimo de luminancia en una región. Adimensional (0 a 1), robusto ante cambios globales de brillo.

In [29]:
import numpy as np
from PIL import Image

def obtener_contraste_michelson(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    Lmax, Lmin = arr.max(), arr.min()
    denom = Lmax + Lmin
    return (Lmax - Lmin) / denom if denom != 0 else 0

In [30]:
print(obtener_contraste_michelson('cute.png'))

1.0


## Contraste local (revisar)

Mide la diferencia de luminancia entre un píxel y su vecindario inmediato. Útil para detectar bordes y texturas.

In [31]:
import numpy as np
from PIL import Image
from scipy.ndimage import uniform_filter

def contraste_local(img_path, ventana=5):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    mu_N = uniform_filter(arr, size=ventana)
    with np.errstate(divide='ignore', invalid='ignore'):
        CL = np.where(mu_N != 0, (arr - mu_N) / mu_N, 0)
    return CL

In [32]:
print(contraste_local('cute.png'))

[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  0.00000000e+00
   0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  1.82190445e-16
   1.82190445e-16  1.82190445e-16]
 ...
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ...  3.12326477e-16
   3.12326477e-16  3.12326477e-16]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -4.68489716e-16
  -4.68489716e-16 -4.68489716e-16]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00 ... -4.68489716e-16
  -4.68489716e-16 -4.68489716e-16]]


## Desviación estándar de intensidad

Dispersión de valores de intensidad en la imagen. Indicador de contraste y variabilidad textural.

In [33]:
import numpy as np
from PIL import Image

def obtener_desv_estandar_intensidad(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    M, N = arr.shape
    mu = np.mean(arr)
    return np.sqrt((1 / (M * N)) * np.sum((arr - mu) ** 2))

In [34]:
print(obtener_desv_estandar_intensidad('cute.png'))

79.8728860124458


# Entropía (Contenido de Información)


## Entropía de Shannon

Mide la incertidumbre o cantidad de información promedio en la distribución de intensidades. Indicador de complejidad.

In [35]:
import numpy as np
from PIL import Image

def obtener_entropia_shannon(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img)
    hist = np.bincount(arr.flatten(), minlength=256)
    prob = hist / hist.sum()
    prob = prob[prob > 0]
    return -np.sum(prob * np.log2(prob))

In [36]:
print(obtener_entropia_shannon('cute.png'))

4.5277220094403114


## Entropía conjunta

Información total de incertidumbre asociada a dos variables (canales o imágenes) consideradas simultáneamente.

In [ ]:
import numpy as np
from PIL import Image

def entropia_conjunta(img1_path, img2_path):
    arr1 = np.array(Image.open(img1_path).convert('L')).flatten()
    arr2 = np.array(Image.open(img2_path).convert('L')).flatten()
    hist2d, _, _ = np.histogram2d(arr1, arr2, bins=256,
                                   range=[[0,256],[0,256]])
    prob = hist2d / hist2d.sum()
    prob = prob[prob > 0]
    return -np.sum(prob * np.log2(prob))

## Entropía local por bloques

Entropía de Shannon calculada en bloques locales de la imagen. Permite analizar variabilidad espacial de la información.

In [37]:
import numpy as np
from PIL import Image

def entropia_local_bloques(img_path, bloque=16):
    img = Image.open(img_path).convert('L')
    arr = np.array(img)
    M, N = arr.shape
    mapa = np.zeros((M // bloque, N // bloque))
    for i in range(M // bloque):
        for j in range(N // bloque):
            b = arr[i*bloque:(i+1)*bloque, j*bloque:(j+1)*bloque]
            hist = np.bincount(b.flatten(), minlength=256)
            prob = hist / hist.sum()
            prob = prob[prob > 0]
            mapa[i, j] = -np.sum(prob * np.log2(prob))
    return mapa

In [38]:
print(entropia_local_bloques('cute.png'))

[[ 1.52260606  1.43188315  0.82932893  0.93885751  0.87743731  0.94016019
   1.45386925  0.98869941  0.98869941  1.30283371  0.16032873 -0.
   0.8463541   0.84030689  0.1388005   0.42817563  0.99365071  0.82932893
   1.10961532  0.43594806  0.89154351  0.6059163   0.88694561  0.32183715
   0.42276234  0.82932893  0.30598487  1.80303355]
 [ 0.954434    0.36708876  0.62526241  0.98869941  0.30598487  1.24150877
   0.91299921  0.09198335  0.81809146  0.68787604  1.31604263  1.03896525
   3.50528273  3.94266124  1.29005869  0.48462525  3.33524737  3.70733634
   3.27446021  1.76974329  0.67939293  0.20062232 -0.         -0.
   0.98231661 -0.         -0.          1.35548757]
 [ 1.12151498  1.17044674  1.07423649  0.87743731  0.95726588  0.32183715
   2.806687    3.46820587  4.0688991   1.5628164   4.30185435  3.73569284
   2.46995439  2.53642847  2.83582249  3.86601144  3.6950503   0.90440821
   1.48758344  3.72235541  1.03163078 -0.         -0.         -0.
   1.03266443 -0.         -0.     

# Frecuencias

## Energía en altas frecuencias

Concentración de potencia espectral en componentes de alta frecuencia. Indica presencia de bordes y detalles nítidos.

In [39]:
import numpy as np
from PIL import Image

def obtener_energia_altas_frecuencias(img_path, radio=0.5):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    F = np.fft.fftshift(np.fft.fft2(arr))
    M, N = arr.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    umbral = radio * min(M, N) / 2
    mask_hf = dist > umbral
    return np.sum(np.abs(F[mask_hf])**2)

In [40]:
print(obtener_energia_altas_frecuencias('cute.png', radio=0.5))

1932853942402.6194


## Energía en bajas frecuencias

Potencia espectral en componentes de baja frecuencia. Corresponde a variaciones suaves, fondos y formas generales.

In [41]:
import numpy as np
from PIL import Image

def obtener_energia_bajas_frecuencias(img_path, radio=0.5):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    F = np.fft.fftshift(np.fft.fft2(arr))
    M, N = arr.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    umbral = radio * min(M, N) / 2
    mask_lf = dist <= umbral
    return np.sum(np.abs(F[mask_lf])**2)

In [ ]:
print(obtener_energia_bajas_frecuencias('cute.png', radio=0.5))

## Relación high/low frequency ratio

Proporción de energía entre altas y bajas frecuencias. Evalúa el nivel de detalle o nitidez frente a la estructura global.

In [42]:
import numpy as np
from PIL import Image

def obtener_hlfr(img_path, radio=0.5):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    F = np.fft.fftshift(np.fft.fft2(arr))
    M, N = arr.shape
    cy, cx = M // 2, N // 2
    Y, X = np.ogrid[:M, :N]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    umbral = radio * min(M, N) / 2
    EHF = np.sum(np.abs(F[dist > umbral])**2)
    ELF = np.sum(np.abs(F[dist <= umbral])**2)
    return EHF / ELF if ELF != 0 else 0

In [44]:
print(obtener_hlfr('cute.png', radio=0.5))

0.0017378561430272308


## Densidad espectral de potencia 

Distribución de la energía de la imagen en el dominio de frecuencias 2D. Se obtiene de la DFT bidimensional.

In [45]:
import numpy as np
from PIL import Image

def psd(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    M, N = arr.shape
    F = np.fft.fft2(arr)
    return (np.abs(F)**2) / (M * N)

In [46]:
print(psd('cute.png'))

[[4.12012088e+09 3.43491795e+07 5.77299015e+06 ... 2.20888293e+07
  5.77299015e+06 3.43491795e+07]
 [3.52282144e+07 2.94737090e+07 9.18670745e+06 ... 8.11809586e+06
  6.43062554e+05 2.75935556e+07]
 [1.13090184e+07 2.20033135e+06 2.48512583e+06 ... 2.61189145e+06
  1.11714361e+06 3.06826940e+06]
 ...
 [8.63441737e+06 1.59661290e+07 2.27033807e+07 ... 1.64655225e+06
  6.55130330e+06 2.63426627e+05]
 [1.13090184e+07 3.06826940e+06 1.11714361e+06 ... 7.59001460e+06
  2.48512583e+06 2.20033135e+06]
 [3.52282144e+07 2.75935556e+07 6.43062554e+05 ... 1.83693244e+07
  9.18670745e+06 2.94737090e+07]]


# Bordes y Gradientes

## Densidad de bordes

Fracción de píxeles detectados como bordes respecto al total de píxeles de la imagen.

In [48]:
import numpy as np
from PIL import Image
from skimage import feature

def obtener_densidad_bordes(img_path, sigma=1.0):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    edges = feature.canny(arr / 255.0, sigma=sigma)
    M, N = arr.shape
    return np.sum(edges) / (M * N)

In [49]:
print(obtener_densidad_bordes('cute.png', sigma=1.0))

0.029116656676851406


## Magnitud media del gradiente

Cambio promedio de intensidad en la imagen. Indica el nivel global de bordes y detalles presentes.

In [50]:
import numpy as np
from PIL import Image
from scipy.ndimage import sobel

def obtener_magnitud_media_gradiente(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    Gx = sobel(arr, axis=1)
    Gy = sobel(arr, axis=0)
    G = np.sqrt(Gx**2 + Gy**2)
    M, N = arr.shape
    return (1 / (M * N)) * np.sum(G)

In [51]:
print(obtener_magnitud_media_gradiente('cute.png'))

34.29010530538024


## Desviación estándar del gradiente

Variabilidad de la magnitud del gradiente. Cuantifica la complejidad estructural de la imagen.

In [52]:
import numpy as np
from PIL import Image
from scipy.ndimage import sobel

def obtener_desv_estandar_gradiente(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    Gx = sobel(arr, axis=1)
    Gy = sobel(arr, axis=0)
    G = np.sqrt(Gx**2 + Gy**2)
    M, N = arr.shape
    mu_G = np.mean(G)
    return np.sqrt((1 / (M * N)) * np.sum((G - mu_G)**2))

In [53]:
print(obtener_desv_estandar_gradiente('cute.png'))

130.71270637896714


## Mapa Sobel

Calcula gradientes horizontales y verticales mediante convolución con máscaras Sobel. Produce bordes gruesos.

In [54]:
import numpy as np
from PIL import Image
from scipy.ndimage import convolve

def mapa_sobel(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64)
    Sx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]])
    Sy = np.array([[-1,-2,-1],[0,0,0],[1,2,1]])
    Gx = convolve(arr, Sx)
    Gy = convolve(arr, Sy)
    G = np.sqrt(Gx**2 + Gy**2)
    return G

In [55]:
print(mapa_sobel('cute.png')[10])

[0.         0.         0.         0.         4.         4.
 4.         4.         0.         0.         0.         0.
 0.         1.41421356 3.16227766 4.         4.         4.24264069
 3.16227766 0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         4.         4.         0.         0.         0.
 0.         0.         2.         2.         0.         1.41421356
 2.82842712 1.41421356 0.         0.         0.         0.
 0.         0.         0.         0.         4.         4.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 4.         4.         0.         0.         0.         0.
 0.         0.         0.         4.         4.         0.
 4.         4.         0.         1.41421356 3.16227766 4.
 4.24264069 3.16227766 0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.    

## Mapa Canny

Detector óptimo multi-etapa: suavizado gaussiano → gradiente → supresión de no-máximos → histéresis.

In [56]:
import numpy as np
from PIL import Image
from skimage import feature

def mapa_canny(img_path, sigma=1.0, low=0.1, high=0.2):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.float64) / 255.0
    edges = feature.canny(arr, sigma=sigma,
                          low_threshold=low, high_threshold=high)
    return edges.astype(np.uint8)

In [59]:
print(mapa_canny('cute.png', sigma=1.0, low=0.1, high=0.2))

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


# Textura

## Haralick features (GLCM)

14 medidas estadísticas de textura de segundo orden basadas en la matriz de co-ocurrencia de niveles de gris.

In [60]:
import numpy as np
from PIL import Image
from skimage.feature import graycomatrix, graycoprops

def haralick_features(img_path, distancias=[1], angulos=[0]):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.uint8)
    glcm = graycomatrix(arr, distances=distancias,
                        angles=angulos, normed=True)
    feats = {}
    for prop in ['contrast','dissimilarity','homogeneity',
                 'energy','correlation','ASM']:
        feats[prop] = graycoprops(glcm, prop)[0, 0]
    return feats

In [61]:
print(haralick_features('cute.png') )

{'contrast': np.float64(243.39397624489632), 'dissimilarity': np.float64(3.097887241399519), 'homogeneity': np.float64(0.8637981663378783), 'energy': np.float64(0.27791547571548497), 'correlation': np.float64(0.9809617972411472), 'ASM': np.float64(0.07723701164216433)}


## Homogeneidad de textura (GLCM)

Qué tan cercanos son los valores de intensidad de píxeles vecinos. Alto = textura suave o uniforme.

In [62]:
import numpy as np
from PIL import Image
from skimage.feature import graycomatrix, graycoprops

def obtener_homogeneidad_textura(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.uint8)
    glcm = graycomatrix(arr, distances=[1], angles=[0], normed=True)
    return graycoprops(glcm, 'homogeneity')[0, 0]

In [63]:
print(obtener_homogeneidad_textura('cute.png') )

0.8637981663378783


## Energía de textura (GLCM Energy / ASM)

Mide la uniformidad global (Angular Second Moment). Alto = textura uniforme o repetitiva.

In [64]:
import numpy as np
from PIL import Image
from skimage.feature import graycomatrix, graycoprops

def obtener_energia_textura(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.uint8)
    glcm = graycomatrix(arr, distances=[1], angles=[0], normed=True)
    return graycoprops(glcm, 'energy')[0, 0]

In [66]:
print(obtener_energia_textura('cute.png') )

0.27791547571548497


## Correlación (GLCM)

Dependencia lineal entre niveles de gris de píxeles vecinos. Alta correlación = patrón estructural consistente.

In [67]:
import numpy as np
from PIL import Image
from skimage.feature import graycomatrix, graycoprops

def obtener_correlacion_glcm(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.uint8)
    glcm = graycomatrix(arr, distances=[1], angles=[0], normed=True)
    return graycoprops(glcm, 'correlation')[0, 0]

In [68]:
print(obtener_correlacion_glcm('cute.png') )

0.9809617972411472


## Contraste de textura (GLCM Contrast)

Variaciones locales de intensidad entre píxeles vecinos. Cuantifica la aspereza o definición de bordes en la textura.

In [69]:
import numpy as np
from PIL import Image
from skimage.feature import graycomatrix, graycoprops

def obtener_contraste_textura(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.uint8)
    glcm = graycomatrix(arr, distances=[1], angles=[0], normed=True)
    return graycoprops(glcm, 'contrast')[0, 0]

In [70]:
print(obtener_contraste_textura('cute.png') )

243.39397624489632


## Entropía de textura (GLCM Entropy)

Grado de aleatoriedad o desorden en la distribución espacial de niveles de gris entre píxeles vecinos.

In [73]:
import numpy as np
from PIL import Image
from skimage.feature import graycomatrix

def obtener_entropia_textura_glcm(img_path):
    img = Image.open(img_path).convert('L')
    arr = np.array(img, dtype=np.uint8)
    glcm = graycomatrix(arr, distances=[1], angles=[0], normed=True)
    P = glcm[:, :, 0, 0]
    P_nz = P[P > 0]
    return -np.sum(P_nz * np.log2(P_nz))

In [ ]:
print(obtener_entropia_textura_glcm('cute.png') )

5.6235409127678455


# Saturación y Crominancia

## Media de saturación

Nivel promedio de pureza cromática en la imagen en espacio HSV.

In [74]:
import numpy as np
from PIL import Image

def obtener_media_saturacion(img_path):
    img = Image.open(img_path).convert('RGB')
    arr = np.array(img, dtype=np.float64) / 255.0
    R, G, B = arr[:,:,0], arr[:,:,1], arr[:,:,2]
    V = np.max(arr, axis=2)
    mn = np.min(arr, axis=2)
    S = np.where(V != 0, (V - mn) / V, 0)
    M, N = S.shape
    return (1 / (M * N)) * np.sum(S)

In [75]:
print(obtener_media_saturacion('cute.png') )

0.3027605989811493


C:\Users\santi\AppData\Local\Temp\ipykernel_5036\2605744044.py:10: RuntimeWarning: invalid value encountered in divide
  S = np.where(V != 0, (V - mn) / V, 0)


## Varianza de saturación

Dispersión de valores de saturación. Indica la uniformidad o variabilidad cromática de la imagen.

In [76]:
import numpy as np
from PIL import Image

def obtener_varianza_saturacion(img_path):
    img = Image.open(img_path).convert('RGB')
    arr = np.array(img, dtype=np.float64) / 255.0
    V = np.max(arr, axis=2)
    mn = np.min(arr, axis=2)
    S = np.where(V != 0, (V - mn) / V, 0)
    M, N = S.shape
    mu_S = np.mean(S)
    return (1 / (M * N)) * np.sum((S - mu_S)**2)

In [77]:
print(obtener_varianza_saturacion('cute.png') )

0.05161811747177644


C:\Users\santi\AppData\Local\Temp\ipykernel_5036\2404986969.py:9: RuntimeWarning: invalid value encountered in divide
  S = np.where(V != 0, (V - mn) / V, 0)


## Varianza de matiz (Hue variance)

Dispersión angular de los tonos de color dominantes. Indica la diversidad cromática en el espacio HSV.

In [81]:
import numpy as np
from PIL import Image

def obtener_varianza_matiz(img_path):
    img = Image.open(img_path).convert('RGB')
    arr = np.array(img, dtype=np.float64) / 255.0
    R, G, B = arr[:,:,0], arr[:,:,1], arr[:,:,2]
    Cmax = np.max(arr, axis=2)
    Cmin = np.min(arr, axis=2)
    delta = Cmax - Cmin
    H = np.zeros_like(R)
    mask_R = (Cmax == R) & (delta != 0)
    mask_G = (Cmax == G) & (delta != 0)
    mask_B = (Cmax == B) & (delta != 0)
    H[mask_R] = 60 * (((G - B) / delta) % 6)[mask_R]
    H[mask_G] = 60 * (((B - R) / delta) + 2)[mask_G]
    H[mask_B] = 60 * (((R - G) / delta) + 4)[mask_B]
    M, N = H.shape
    mu_H = np.mean(H)
    return (1 / (M * N)) * np.sum((H - mu_H)**2)

In [82]:
print(obtener_varianza_matiz('cute.png'))

2859.9755887941164


C:\Users\santi\AppData\Local\Temp\ipykernel_5036\330743622.py:15: RuntimeWarning: invalid value encountered in divide
  H[mask_R] = 60 * (((G - B) / delta) % 6)[mask_R]
C:\Users\santi\AppData\Local\Temp\ipykernel_5036\330743622.py:16: RuntimeWarning: invalid value encountered in divide
  H[mask_G] = 60 * (((B - R) / delta) + 2)[mask_G]
C:\Users\santi\AppData\Local\Temp\ipykernel_5036\330743622.py:17: RuntimeWarning: invalid value encountered in divide
  H[mask_B] = 60 * (((R - G) / delta) + 4)[mask_B]


## Distribución en espacio CIELAB

Dispersión de colores en el espacio perceptual Lab*, diseñado para coincidir con la visión humana.

In [85]:
import numpy as np
from PIL import Image
from skimage.color import rgb2lab

def obtener_distribucion_cielab(img_path):
    img = Image.open(img_path).convert('RGB')
    arr = np.array(img, dtype=np.float64) / 255.0
    lab = rgb2lab(arr)
    L, a, b = lab[:,:,0], lab[:,:,1], lab[:,:,2]
    M, N = L.shape
    return {
        'media_a': np.mean(a), 'media_b': np.mean(b),
        'varianza_a': np.mean((a - np.mean(a))**2),
        'varianza_b': np.mean((b - np.mean(b))**2)
    }

In [87]:
print(obtener_distribucion_cielab('cute.png'))

{'media_a': np.float64(-13.976138977089333), 'media_b': np.float64(19.083287794225754), 'varianza_a': np.float64(355.42075145742155), 'varianza_b': np.float64(190.34058338317703)}


In [86]:
cielab_media_a, cielab_media_b, cielab_varianza_a, cielab_varianza_b = obtener_distribucion_cielab('cute.png').values()
print(cielab_media_a, cielab_media_b, cielab_varianza_a, cielab_varianza_b)

-13.976138977089333 19.083287794225754 355.42075145742155 190.34058338317703


# Resolución Espacial

## Megapíxeles

Resolución total de la imagen expresada en millones de píxeles.

In [88]:
from PIL import Image

def obtener_megapixeles(img_path):
    img = Image.open(img_path)
    M, N = img.size[1], img.size[0]
    return (M * N) / 1e6

In [89]:
print(obtener_megapixeles('cute.png'))

0.205209


## Aspect ratio

Proporción entre el ancho y el alto de la imagen, expresada como razón W/H.

In [91]:
from PIL import Image

def obtener_aspect_ratio(img_path):
    img = Image.open(img_path)
    W, H = img.size
    return W / H

In [92]:
print(obtener_aspect_ratio('cute.png'))

1.0
